In [1]:
!pip install "datasets<3.0.0" soundfile transformers accelerate torch resemblyzer librosa pandas scipy parler_tts

In [2]:
!pip install protobuf==4.25.9

In [3]:
import os
import numpy as np
import torch
import soundfile as sf
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav

def run_evaluation(dataset_name, samples_list, text_key):
    """
    Evaluates TTS performance by comparing generated audio to reference audio.
    """
    base_dir = f"mushra_{dataset_name}"
    for sub in ["reference", "generated"]:
        os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

    results = []
    for i in range(len(samples_list)):
        try:
            sample = samples_list[i]
            text = sample[text_key]
            ref_array = np.array(sample['audio']['array']).astype(np.float32)
            ref_sr = sample['audio']['sampling_rate']

            if ref_array.ndim > 1:
                ref_array = ref_array.mean(axis=1)

            ref_path = f"{base_dir}/reference/sample_{i}.wav"
            gen_path = f"{base_dir}/generated/sample_{i}.wav"

            sf.write(ref_path, ref_array, ref_sr)

            # ================== NEW: Indic-Parler-TTS Generation ==================
            description_input_ids = description_tokenizer(
                MALE_URDU_DESCRIPTION, return_tensors="pt"
            ).to(device)
            prompt_input_ids = tokenizer(text, return_tensors="pt").to(device)

            with torch.no_grad():
                generation = model.generate(
                    input_ids=description_input_ids.input_ids,
                    attention_mask=description_input_ids.attention_mask,
                    prompt_input_ids=prompt_input_ids.input_ids,
                    prompt_attention_mask=prompt_input_ids.attention_mask,
                    do_sample=True,
                    temperature=0.8
                )

            gen_speech = generation.cpu().numpy().squeeze()
            gen_sr = model.config.sampling_rate
            sf.write(gen_path, gen_speech, gen_sr)

            # Embedding Similarity (Resemblyzer)
            wav_ref = preprocess_wav(Path(ref_path))
            wav_gen = preprocess_wav(Path(gen_path))

            mid = len(wav_ref) // 2
            emb_ref_full = encoder.embed_utterance(wav_ref)
            emb_ref_a = encoder.embed_utterance(wav_ref[:mid])
            emb_ref_b = encoder.embed_utterance(wav_ref[mid:])
            emb_gen = encoder.embed_utterance(wav_gen)

            ref_gen_sim = np.dot(emb_ref_full, emb_gen) / (np.linalg.norm(emb_ref_full) * np.linalg.norm(emb_gen))
            ref_self_sim = np.dot(emb_ref_a, emb_ref_b) / (np.linalg.norm(emb_ref_a) * np.linalg.norm(emb_ref_b))

            results.append({
                "sample_id": i,
                "urdu_text": text,
                "audio_duration_sec": round(len(ref_array)/ref_sr, 2),
                "ref_gen_similarity": float(ref_gen_sim),
                "ref_self_similarity": float(ref_self_sim)
            })
            print(f"[{dataset_name.upper()}] Sample {i+1}: sim={ref_gen_sim:.4f} | self={ref_self_sim:.4f}")

        except Exception as e:
            print(f"Error in {dataset_name} sample {i}: {e}")
            continue

    return results

In [9]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
from resemblyzer import VoiceEncoder

from google.colab import userdata
try:
    hf_token = userdata.get("HF_TOKEN")
    print("✅ HF token loaded from Colab secrets")
except Exception:
    hf_token = None
    print("⚠️  HF_TOKEN not found in secrets — proceeding without token (may fail for gated datasets)")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Indic-Parler-TTS model to {device}...")

# ================== NEW: Indic-Parler-TTS Loading ==================
# Explicitly passing the token to handle gated repo access
model = ParlerTTSForConditionalGeneration.from_pretrained(
    "ai4bharat/indic-parler-tts",
    token=hf_token
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "ai4bharat/indic-parler-tts",
    token=hf_token
)

description_tokenizer = AutoTokenizer.from_pretrained(
    model.config.text_encoder._name_or_path,
    token=hf_token
)

# Fixed male Urdu description using "Rohit"
MALE_URDU_DESCRIPTION = (
    "Rohit's voice is clear and natural with a moderate speed and pitch. "
    "The recording is of very high quality, with the speaker's voice sounding very close up "
    "and no background noise."
)

# Initialize Resemblyzer encoder
encoder = VoiceEncoder()

print("✅ Indic-Parler-TTS loaded successfully (male Urdu voice: Rohit)")

✅ HF token loaded from Colab secrets
Loading Indic-Parler-TTS model to cuda...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

generation_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loaded the voice encoder model on cuda in 0.18 seconds.
✅ Indic-Parler-TTS loaded successfully (male Urdu voice: Rohit)


In [10]:
NUM_SAMPLES = 30
from datasets import load_dataset
# Use streaming=True to fetch samples instantly
ds_fleurs_stream = load_dataset("google/fleurs", "ur_pk", split="test", trust_remote_code=True, streaming=True)

# Take first 15 samples from the stream
fleurs_samples = []
for sample in ds_fleurs_stream:
    fleurs_samples.append(sample)
    if len(fleurs_samples) >= NUM_SAMPLES:
        break

results_fleurs = run_evaluation("fleurs", fleurs_samples, "transcription")

[FLEURS] Sample 1: sim=0.5393 | self=0.9231
[FLEURS] Sample 2: sim=0.6229 | self=0.8647
[FLEURS] Sample 3: sim=0.4854 | self=0.9433
[FLEURS] Sample 4: sim=0.4937 | self=0.8513
[FLEURS] Sample 5: sim=0.5345 | self=0.8958
[FLEURS] Sample 6: sim=0.4527 | self=0.8025
[FLEURS] Sample 7: sim=0.5242 | self=0.8967
[FLEURS] Sample 8: sim=0.5140 | self=0.9305
[FLEURS] Sample 9: sim=0.4501 | self=0.9059
[FLEURS] Sample 10: sim=0.5255 | self=0.9462
[FLEURS] Sample 11: sim=0.5111 | self=0.8696
[FLEURS] Sample 12: sim=0.5154 | self=0.9473
[FLEURS] Sample 13: sim=0.5101 | self=0.8265
[FLEURS] Sample 14: sim=0.4720 | self=0.9052
[FLEURS] Sample 15: sim=0.6105 | self=0.9287
[FLEURS] Sample 16: sim=0.4668 | self=0.9214
[FLEURS] Sample 17: sim=0.5450 | self=0.8967
[FLEURS] Sample 18: sim=0.5179 | self=0.8300
[FLEURS] Sample 19: sim=0.5724 | self=0.9198
[FLEURS] Sample 20: sim=0.4634 | self=0.7896
[FLEURS] Sample 21: sim=0.4861 | self=0.9198
[FLEURS] Sample 22: sim=0.5661 | self=0.8938
[FLEURS] Sample 23:

In [11]:
import pandas as pd
import zipfile
import os

# 1. Prepare the data for CSV
all_results = []
# Assuming results_fleurs exists from previous execution
try:
    for entry in results_fleurs:
        all_results.append({
            "dataset": "fleurs",
            "sample_id": entry["sample_id"],
            "urdu_text": entry["urdu_text"],
            "audio_duration_sec": entry["audio_duration_sec"],
            "ref_gen_similarity": entry["ref_gen_similarity"]
        })
except NameError:
    print("Warning: results_fleurs not found. Please ensure the evaluation cell has finished running.")

# 2. Save to CSV
csv_filename = "evaluation_results.csv"
df = pd.DataFrame(all_results)
df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
print(f"Created {csv_filename}")

# 3. Create Zip file
zip_filename = "mushra_results_package.zip"
datasets_to_zip = ["fleurs", "fleurs_male"] # Add others if they were generated

with zipfile.ZipFile(zip_filename, "w") as zf:
    # Add the CSV
    zf.write(csv_filename)

    # Add the audio folders
    for d_name in datasets_to_zip:
        folder = f"mushra_{d_name}"
        if os.path.exists(folder):
            for root, dirs, files in os.walk(folder):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Preserve folder structure inside zip
                    zf.write(file_path, os.path.relpath(file_path, "."))

print(f"Successfully created {zip_filename} containing results and audio samples.")

Created evaluation_results.csv
Successfully created mushra_results_package.zip containing results and audio samples.
